# Quantizing Jina embeddings v5 with BBQ

In this notebook, we will compress **Jina embeddings v5** with **Better Binary Quantization (BBQ)** in Elasticsearch and measure two things on multilingual data:

- **Memory**: how much RAM the vectors use with and without BBQ.
- **Recall**: whether BBQ returns the same results as full precision.

Elasticsearch stores [4 bytes per dimension](https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/dense-vector) for `float` vectors, so a single 1024-dimension vector takes about 4 KB. HNSW [needs the vector data in memory](https://www.elastic.co/docs/deploy-manage/production-guidance/optimize-performance/approximate-knn-search) to search efficiently, meaning that 4 KB sits in RAM. With a few million documents, vectors become the largest cost in the cluster.

## Setup
### Install dependencies

In [ ]:
%pip install -q elasticsearch==9.4 datasets==5.0 numpy==2.2.5 matplotlib==3.10 python-dotenv==1.0


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os
import numpy as np

load_dotenv()

ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")

### Connect to Elasticsearch

You will need an Elasticsearch deployment with the Elastic-managed Jina v5 inference endpoint available, and a `.env` file with `ELASTICSEARCH_URL` and `ELASTICSEARCH_API_KEY`.

In [3]:
from elasticsearch import Elasticsearch, helpers

es_client = Elasticsearch(
    ELASTICSEARCH_URL, api_key=ELASTICSEARCH_API_KEY, request_timeout=120
)
es_client.info()

ObjectApiResponse({'name': 'instance-0000000001', 'cluster_name': 'fd31ad5f77d841d69b622c17ed64b766', 'cluster_uuid': 'YxmDkf9DSwWpvcCLv98q8A', 'version': {'number': '9.4.1', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '3c7c6027c5769d860d87448e2749f4c550a239da', 'build_date': '2026-05-08T10:08:29.383338563Z', 'build_snapshot': False, 'lucene_version': '10.4.0', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

### Create the two indices

We will create two indices with the same mapping. The only difference is the `index_options.type`:

| Index | `index_options` | What it stores |
|---|---|---|
| `vectors-float32` | `hnsw` | raw `float32` (baseline) |
| `vectors-bbq` | `bbq_hnsw` | 1-bit BBQ + corrective factors |

Both use `jina-embeddings-v5-text-small`'s 1024 dimensions and cosine similarity.

In [4]:
DIMS = 1024
FLOAT_INDEX = "vectors-float32"
BBQ_INDEX = "vectors-bbq"


def create_index(name, index_options):
    if es_client.indices.exists(index=name):
        es_client.indices.delete(index=name)
    es_client.indices.create(
        index=name,
        mappings={
            "properties": {
                "text": {"type": "text"},
                "lang": {"type": "keyword"},
                "embedding": {
                    "type": "dense_vector",
                    "dims": DIMS,
                    "index": True,
                    "similarity": "cosine",
                    "index_options": index_options,
                },
            }
        },
    )


create_index(FLOAT_INDEX, {"type": "hnsw"})  # raw float32 baseline
create_index(BBQ_INDEX, {"type": "bbq_hnsw"})  # 1-bit BBQ
print(f"Created '{FLOAT_INDEX}' (float32) and '{BBQ_INDEX}' (BBQ)")

Created 'vectors-float32' (float32) and 'vectors-bbq' (BBQ)


## 1. What is quantization?

An embedding is a list of numbers. By default, each number is a `float32`: 4 bytes. Quantization stores each number with fewer bits, trading some precision for space.

Two ways to picture it:

- **JPEG.** A quantized vector is like a JPEG of the original: a smaller, lower-fidelity copy that still gets the job done.
- **Rounding.** It is like writing a weight as `2.8 kg` instead of `2.7563 kg`. Fewer digits, less space, still useful.

| Precision | Bytes / dim | 1024-d vector | Compression |
|---|---|---|---|
| `float32` | 4 | 4096 B | 1x |
| `int8` | 1 | 1024 B | 4x |
| `int4` | 0.5 | 512 B | 8x |
| `binary` (1 bit) | 0.125 | 128 B | **32x** |

The bytes per dimension come from the Elasticsearch [`dense_vector` element types](https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/dense-vector). The diagram below shows the same number stored at each precision and the resulting size per vector.

> New to the idea? IBM's [What is quantization?](https://www.ibm.com/think/topics/quantization) is a good plain-English primer. For Elasticsearch specifics, see [Scalar quantization 101](https://www.elastic.co/search-labs/blog/scalar-quantization-101), [Optimized Scalar Quantization](https://www.elastic.co/search-labs/blog/optimized-scalar-quantization-elasticsearch), and the [BBQ deep dive](https://www.elastic.co/search-labs/blog/better-binary-quantization-lucene-elasticsearch).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: the same number snapped to fewer and fewer levels
ax = axes[0]
levels = {
    "float32\n(~continuous)": 4096,
    "int8\n(256 levels)": 256,
    "int4\n(16 levels)": 16,
    "binary\n(2 levels)": 2,
}
val = 0.42  # the original float value we want to store
for i, (name, n) in enumerate(levels.items()):
    y = len(levels) - i
    ax.hlines(y, -1, 1, color="#e8e8e8", lw=8, zorder=1)
    if n <= 16:
        ticks = np.linspace(-1, 1, n)
        ax.scatter(ticks, [y] * n, color="#9aa0a6", s=18, zorder=2)
        snapped = ticks[np.argmin(np.abs(ticks - val))]
    else:
        snapped = val
    ax.scatter([snapped], [y], color="#0b64dd", s=110, zorder=3)
    ax.text(-1.18, y, name, ha="right", va="center", fontsize=9)
ax.axvline(val, color="#34a853", ls="--", lw=1)
ax.text(
    val, len(levels) + 0.6, "original value", color="#34a853", ha="center", fontsize=8
)
ax.set_xlim(-1.5, 1.15)
ax.set_ylim(0.3, len(levels) + 0.9)
ax.axis("off")
ax.set_title("Value stored at each precision")

# Right: bytes per 1024-d vector
ax = axes[1]
names = ["float32", "int8", "int4", "binary"]
bytes_per = [4096, 1024, 512, 128]
comp = [1, 4, 8, 32]
colors = ["#0b64dd", "#4285f4", "#8ab4f8", "#c6dafc"]
bars = ax.barh(names[::-1], bytes_per[::-1], color=colors[::-1])
ax.set_xlabel("Bytes per 1024-d vector")
ax.set_title("Bytes per vector")
for b, by, c in zip(bars, bytes_per[::-1], comp[::-1]):
    ax.text(
        by + 60,
        b.get_y() + b.get_height() / 2,
        f"{by} B  ({c}x)",
        va="center",
        fontsize=9,
    )
ax.set_xlim(0, 5200)

plt.tight_layout()
plt.show()

### Why BBQ and not plain binary?

Plain 1-bit quantization loses too much accuracy. BBQ keeps 1-bit vectors precise through three mechanisms:

1. **Asymmetric precision.** Stored vectors use 1 bit per dimension; the query vector is quantized to int4. 
2. **Corrective factors.** A few floats per vector record the rounding error and correct distances at scoring time.
3. **Oversample and rescore.** BBQ scans candidates with the bits, then re-ranks the top ones with higher precision. Fetching the top 10 means scanning about 30 candidates.

The result is vectors that are about 32x smaller, with ranking quality close to full precision. The next sections measure the memory saving and the recall.

## 2. How Jina embeddings v5 works

There are three things that matter for this blog post:

- **One model, many tasks.** Instead of separate models, v5 uses small **LoRA adapters** on top of a single base model, one for each task: *retrieval*, *text-matching*, *clustering*, and *classification*. Elasticsearch picks the right adapter automatically at index and query time.
- **Matryoshka dimensions.** v5 is trained so you can **truncate** the vector (1024 → 512 → 256) and keep most of the quality. This is another way to shrink vectors, independent of quantization.
- **Quantization-aware training.** v5 is trained to work with BBQ, so its 1-bit vectors lose little accuracy.

We use `jina-embeddings-v5-text-small`: 1024 dims, 32k token context, multilingual across 211 languages. That puts it above the [384-dim floor](https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/bbq) below which BBQ starts to lose accuracy.

> Full model details: [jina-embeddings-v5-text on Search Labs](https://www.elastic.co/search-labs/blog/jina-embeddings-v5-text).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: one base model + task-specific LoRA adapters
ax = axes[0]
ax.axis("off")
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.add_patch(
    FancyBboxPatch(
        (3.3, 4.2), 3.4, 1.5, boxstyle="round,pad=0.1", fc="#0b64dd", ec="none"
    )
)
ax.text(
    5,
    4.95,
    "Jina v5\nbase model",
    color="white",
    ha="center",
    va="center",
    fontsize=10,
    weight="bold",
)
tasks = ["retrieval", "text-matching", "clustering", "classification"]
for t, x in zip(tasks, [1.2, 3.7, 6.3, 8.8]):
    ax.add_patch(
        FancyBboxPatch(
            (x - 0.95, 7.6),
            1.9,
            1.1,
            boxstyle="round,pad=0.08",
            fc="#e8f0fe",
            ec="#0b64dd",
        )
    )
    ax.text(x, 8.15, t, ha="center", va="center", fontsize=8)
    ax.annotate(
        "",
        xy=(x, 7.55),
        xytext=(5, 5.75),
        arrowprops=dict(arrowstyle="->", color="#9aa0a6"),
    )
ax.text(
    5,
    3.3,
    "one LoRA adapter per task\n(auto-selected by Elasticsearch)",
    ha="center",
    fontsize=8,
    color="#555",
)
ax.set_title("LoRA adapter per task")

# Right: Matryoshka nested dimensions
ax = axes[1]
ax.axis("off")
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
for dim, color, w in [
    (1024, "#0b64dd", 8.6),
    (512, "#4285f4", 6.0),
    (256, "#8ab4f8", 3.4),
]:
    h = w * 0.5
    ax.add_patch(Rectangle((5 - w / 2, 5 - h / 2), w, h, fc="none", ec=color, lw=2.2))
    ax.text(
        5,
        5 - h / 2 + 0.28,
        f"{dim}d",
        color=color,
        ha="center",
        va="bottom",
        fontsize=9,
        weight="bold",
    )
ax.set_title("Matryoshka dimensions")
ax.text(
    5,
    0.6,
    "cut 1024 → 512 → 256 with minimal quality loss",
    ha="center",
    fontsize=8,
    color="#555",
)

plt.tight_layout()
plt.show()

## 3. Run the experiment

We will embed a multilingual news corpus once and index the same vectors into both indices. That way, the only difference between them is the storage format.

### Point at the Jina v5 inference endpoint

`jina-embeddings-v5-text-small` is available as an Elastic-managed (EIS) inference endpoint. We call it directly to turn text into vectors.

In [7]:
INFERENCE_ID = ".jina-embeddings-v5-text-small"


def embed(texts, batch_size=16):
    """Turn a list of strings into a float32 numpy array of embeddings.

    The Jina v5 EIS endpoint accepts at most 16 inputs per request.
    """
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        try:
            resp = es_client.inference.text_embedding(
                inference_id=INFERENCE_ID, input=batch
            )
        except AttributeError:  # older client versions
            resp = es_client.inference.inference(inference_id=INFERENCE_ID, input=batch)
        out.extend(item["embedding"] for item in resp["text_embedding"])
    return np.array(out, dtype=np.float32)


# Smoke test: one short text should come back as a 1024-d vector
embed(["hello world"]).shape

(1, 1024)

### Load a multilingual news dataset

We will stream real news articles from [`hotchpotch/multilingual_cc_news`](https://huggingface.co/datasets/hotchpotch/multilingual_cc_news), a parquet mirror of CC-News. We take about 1,000 articles from five languages (around 5,000 docs total), plus a small held-out set of headlines to use as search queries. Using multiple languages also lets Jina v5 show its multilingual strength.

In [8]:
from datasets import load_dataset

LANGS = ["en", "de", "ja", "pt", "ru"]
PER_LANG_DOCS = 1000
PER_LANG_QUERIES = 20

docs, queries = [], []
for lang in LANGS:
    ds = load_dataset(
        "hotchpotch/multilingual_cc_news", lang, split="train", streaming=True
    )
    rows = [
        r
        for r in ds.take(PER_LANG_DOCS + PER_LANG_QUERIES)
        if r.get("maintext") and r.get("title")
    ]
    for row in rows[:PER_LANG_DOCS]:
        text = (row["title"] + ". " + row["maintext"]).replace("\n", " ").strip()
        docs.append({"text": text[:1000], "lang": lang})
    for row in rows[PER_LANG_DOCS:]:
        queries.append(
            {"text": row["title"], "lang": lang}
        )  # headlines make natural queries

print(f"Corpus: {len(docs)} docs | Queries: {len(queries)}")

/Users/jeffreyrengifo/.pyenv/versions/3.13.7/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Corpus: 3102 docs | Queries: 18


### Embed once, reuse everywhere

We embed the corpus a single time and feed those exact vectors into both indices. Same vectors in, only the storage format differs.

In [9]:
doc_vectors = embed([d["text"] for d in docs])
query_vectors = embed([q["text"] for q in queries])
doc_vectors.shape, query_vectors.shape

((3102, 1024), (18, 1024))

### Index the identical vectors

We bulk-index the same precomputed vectors into both indices, then force-merge to a single segment so the storage numbers are stable and comparable.

In [10]:
def index_docs(name):
    actions = (
        {
            "_index": name,
            "_id": i,
            "_source": {
                "text": d["text"],
                "lang": d["lang"],
                "embedding": doc_vectors[i].tolist(),
            },
        }
        for i, d in enumerate(docs)
    )
    helpers.bulk(es_client, actions, refresh=True)


for name in (FLOAT_INDEX, BBQ_INDEX):
    index_docs(name)
    es_client.indices.forcemerge(index=name, max_num_segments=1)
    es_client.indices.refresh(index=name)
print("Indexed identical vectors into both indices")

Indexed identical vectors into both indices


## 4. Results: disk vs. memory

The [disk usage API](https://www.elastic.co/docs/api/doc/elasticsearch/operation/operation-indices-disk-usage) reports how many bytes each index spends on vectors (`knn_vectors`).

On disk, the two indices are about the same size. A quantized index still keeps the raw `float32` vectors (needed for rescoring and re-quantization during merges) and adds the 1-bit vectors on top, so BBQ ends up slightly larger.

The real saving is in memory. The HNSW scan only needs the 1-bit vectors in RAM; the raw floats are read from disk to rescore the top candidates. We size that footprint using the documented [kNN memory formulas](https://www.elastic.co/docs/deploy-manage/production-guidance/optimize-performance/approximate-knn-search): `float` uses `num_vectors × dims × 4` and `bbq` uses `num_vectors × (dims/8 + 14)`.

Since both indices store the same raw vectors, the extra bytes on disk in the BBQ index should match the 1-bit payload. We print that as a sanity check.

In [ ]:
def vector_disk_bytes(name):
    du = es_client.indices.disk_usage(index=name, run_expensive_tasks=True)
    field = du[name]["fields"]["embedding"]
    knn = field.get("knn_vectors")
    # Newer ES returns a human-readable string ("12.3mb") plus a sibling
    # *_in_bytes integer; older versions nest {"size_in_bytes": ...}.
    if isinstance(knn, dict):
        return knn["size_in_bytes"]
    return field["knn_vectors_in_bytes"]


float_disk = vector_disk_bytes(FLOAT_INDEX)
bbq_disk = vector_disk_bytes(BBQ_INDEX)

# Resident-memory footprint, from Elasticsearch's documented kNN memory
# formulas: float -> num_vectors * dims * 4 ; bbq -> num_vectors * (dims/8 + 14).
# https://www.elastic.co/docs/deploy-manage/production-guidance/optimize-performance/approximate-knn-search
N = len(docs)
float_mem = N * DIMS * 4
bbq_mem = N * (DIMS // 8 + 14)

print(
    f"On disk    -> float32: {float_disk / 1e6:6.2f} MB | BBQ: {bbq_disk / 1e6:6.2f} MB  (BBQ also stores raw floats for rescoring)"
)
print(
    f"In memory  -> float32: {float_mem / 1e6:6.2f} MB | BBQ: {bbq_mem / 1e6:6.2f} MB  ({float_mem / bbq_mem:.0f}x smaller)"
)
print(
    f"Sanity check: BBQ's extra bytes on disk = {(bbq_disk - float_disk) / 1e6:.2f} MB vs computed 1-bit payload {bbq_mem / 1e6:.2f} MB"
)

In [ ]:
import matplotlib.pyplot as plt

groups = ["On disk", "In memory\n(resident for search)"]
float_vals = [float_disk / 1e6, float_mem / 1e6]
bbq_vals = [bbq_disk / 1e6, bbq_mem / 1e6]

x = np.arange(len(groups))
w = 0.35

fig, ax = plt.subplots(figsize=(7.5, 4.5))
b1 = ax.bar(x - w / 2, float_vals, w, label="float32 (hnsw)", color="#9aa0a6")
b2 = ax.bar(x + w / 2, bbq_vals, w, label="BBQ (bbq_hnsw)", color="#0b64dd")
ax.set_ylabel("Vector storage (MB)")
ax.set_title("Vector storage: disk vs. memory")
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.legend()
for bars in (b1, b2):
    for b in bars:
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height(),
            f"{b.get_height():.1f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
ax.set_ylim(0, max(float_vals + bbq_vals) * 1.15)
plt.tight_layout()
plt.show()

## 5. Results: recall

To check quality, we compare BBQ against full precision. We take the `float32` top-k as the reference and measure how many of those documents BBQ also returns, varying the oversampling factor (`num_candidates / k`).

`recall@k = | BBQ top-k ∩ float32 top-k | / k`, averaged over all queries.

On this corpus, BBQ matches `float32` almost exactly even at 1x oversampling, so the curve is flat. Oversampling helps more on harder, denser datasets where 1-bit collisions are more common.

In [13]:
def search_ids(index, qvec, k=10, num_candidates=10):
    resp = es_client.search(
        index=index,
        size=k,
        _source=False,
        knn={
            "field": "embedding",
            "query_vector": qvec.tolist(),
            "k": k,
            "num_candidates": num_candidates,
        },
    )
    return [h["_id"] for h in resp["hits"]["hits"]]


K = 10

# Ground truth: full-precision float32 with a wide candidate list (~exact)
ground_truth = [
    set(search_ids(FLOAT_INDEX, qv, k=K, num_candidates=2000)) for qv in query_vectors
]

oversamples = [1, 2, 3, 5, 10]
recalls = []
for f in oversamples:
    num_candidates = max(K * f, K)
    hits = 0
    for gt, qv in zip(ground_truth, query_vectors):
        got = set(search_ids(BBQ_INDEX, qv, k=K, num_candidates=num_candidates))
        hits += len(got & gt)
    recalls.append(hits / (len(query_vectors) * K))
    print(f"oversample {f:>2}x -> recall@{K} = {recalls[-1]:.3f}")

oversample  1x -> recall@10 = 0.989
oversample  2x -> recall@10 = 0.989
oversample  3x -> recall@10 = 0.989
oversample  5x -> recall@10 = 0.989
oversample 10x -> recall@10 = 0.989


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(oversamples, recalls, marker="o", color="#0b64dd", lw=2, label="BBQ (bbq_hnsw)")
ax.axhline(1.0, ls="--", color="#9aa0a6", label="float32 baseline")
ax.set_xlabel("Oversample factor  (num_candidates / k)")
ax.set_ylabel(f"Recall@{K} vs float32")
ax.set_title(f"Recall@{K} vs oversample factor")
ax.set_ylim(min(recalls) - 0.05, 1.03)
ax.set_xticks(oversamples)
ax.legend()
plt.tight_layout()
plt.show()

## Cleanup

In [ ]:
for name in (FLOAT_INDEX, BBQ_INDEX):
    if es_client.indices.exists(index=name):
        es_client.indices.delete(index=name)
print("Deleted indices")